# Redes Neuronales 2026

---

# Guía de ejercicios N°10: **Autoencoder** con Fashion-MNIST

Esta guía desarrolla un **Autoencoder** completamente conectado.

El objetivo del **autoencoder** es aprender

$$
x \longrightarrow \hat{x},
$$

de modo que

$$
\hat{x}\approx x.
$$

La propia imagen funciona como entrada y como objetivo.

En las celdas de código se encuentran esqueletos, comentarios y marcadores `# COMPLETAR`.

Trabajar de manera secuencial. Antes de escribir código, intentar anticipar:
- qué forma debe tener cada tensor;
- qué operación conceptual se está implementando;
- qué conjunto de datos corresponde utilizar en cada etapa.

## Ejercicio 1) Preparar el entorno

### 1.1) Importar librerías



In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

### 1.2) Reproducibilidad y dispositivo

Fijamos semillas para facilitar la reproducción del experimento y seleccionamos GPU si está disponible.

Observar especialmente la variable `device`: más adelante tanto el **modelo** como los *batches* deberán encontrarse en el mismo dispositivo.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Dispositivo:", device)

## Ejercicio 2) Cargar y preparar Fashion-MNIST

### 2.1) Transformación

Queremos convertir cada imagen de Fashion-MNIST en un tensor con forma `(1,28,28)` y valores aproximadamente en `[0,1]`.

Esto es coherente con la activación de salida que utilizaremos más adelante:

$$
0\leq x\leq1,
\qquad
0\leq\hat{x}\leq1.
$$

**Tarea:** definir una transformación de `torchvision` que convierta las imágenes a tensores.

**Ayuda:** para esta guía basta una única transformación.

In [ ]:
# COMPLETAR:
# Queremos una transformación que convierta cada imagen a tensor, con valores en [0,1].

transform = transforms.ToTensor()

### 2.2) *Train*, *validation* y *test*

Fashion-MNIST contiene:
- 60.000 imágenes en el conjunto oficial de **entrenamiento**;
- 10.000 imágenes en el conjunto oficial de *test*.

En esta guía dividiremos las 60.000 primeras en:
- 50.000 para *train*;
- 10.000 para *validation*.

Las 10.000 imágenes oficiales de test deben quedar intactas hasta la evaluación final.

**Tarea:**
1. cargar el conjunto oficial de **entrenamiento**;
2. cargar el conjunto oficial de *test*;
3. crear un generador reproducible usando `SEED`;
4. dividir el conjunto oficial de entrenamiento en 50.000 y 10.000 ejemplos.

**Ayudas:**
- utilizar `datasets.FashionMNIST`;
- utilizar `random_split`;
- aplicar la transformación definida en 2.1.

In [ ]:
# COMPLETAR:
# 1) cargar el conjunto oficial de entrenamiento en full_train_set
# 2) cargar el conjunto oficial de test en test_set
# 3) crear split_generator con semilla SEED
# 4) dividir full_train_set en train_set y valid_set

full_train_set = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

test_set = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

split_generator = torch.Generator().manual_seed(SEED)

train_set, valid_set = random_split(
    full_train_set,
    [50_000, 10_000],
    generator=split_generator
)

print("------------------")
print("Train:", len(train_set))
print("Validation:", len(valid_set))
print("Test:", len(test_set))

100%|██████████| 26.4M/26.4M [00:02<00:00, 8.86MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 142kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.67MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 28.1MB/s]

------------------
Train: 50000
Validation: 10000
Test: 10000


### 2.3) Inspeccionar un ejemplo

Cada elemento de Fashion-MNIST se entrega como un par:

```text
(image, label)
```

El **Autoencoder** utilizará la imagen como entrada y objetivo, pero no necesitará la etiqueta durante el entrenamiento.

**Tarea:** tomar el primer ejemplo de `train_set` y mostrar:
- la forma de la imagen;
- su etiqueta;
- el valor mínimo;
- el valor máximo.

**Resultado esperado:** la forma debe ser compatible con una imagen de un solo canal y tamaño 28×28.

In [ ]:
# COMPLETAR:
# Extraer un ejemplo del conjunto de entrenamiento y analizar su contenido.

image, label = train_set[0]

print("Shape:", image.shape)
print("Etiqueta:", label)
print("Mínimo:", image.min().item())
print("Máximo:", image.max().item())

Shape: torch.Size([1, 28, 28])
Etiqueta: 9
Mínimo: 0.0
Máximo: 1.0


### 2.4) Mosaico 3×3 de imágenes con sus etiquetas

Antes de construir el **Autoencoder** conviene observar ejemplos concretos del *dataset*.

Aunque las etiquetas **no se utilizarán para entrenar el autoencoder**, aquí son útiles para reconocer las clases.

El **diccionario de nombres** se entrega completo. La tarea consiste en:
1. seleccionar nueve índices de manera reproducible;
2. crear una figura 3×3;
3. recuperar imagen y etiqueta para cada índice;
4. mostrar cada imagen en escala de grises;
5. colocar como título el nombre de la clase;
6. ocultar los ejes.

**Ayuda:** se puede usar `torch.randperm(...)` y `axes.flat`.

In [ ]:
class_names = {
    0: "T-shirt/top",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle boot",
}

display_generator = torch.Generator().manual_seed(SEED)

# COMPLETAR: seleccionar 9 índices reproducibles.
indices = ...

# COMPLETAR: crear una figura de 3 filas por 3 columnas.
fig, axes = ...

for ax, idx in zip(axes.flat, indices):
    # COMPLETAR: recuperar imagen y etiqueta.
    image, label = ...

    # COMPLETAR: mostrar imagen, título y ocultar ejes.
    ...

plt.tight_layout()
plt.show()

## Ejercicio 3) Crear los `DataLoader`s

Trabajaremos con *batches* de 100 imágenes.

Recordar:
- *train* debe mezclar los ejemplos;
- *validation* y *test* no necesitan mezclarse;
- un *batch* de imágenes tendrá forma `(B,1,28,28)`.

**Tarea:** crear `train_loader`, `valid_loader` y `test_loader`. Después, extraer un *batch* de *train* y comprobar las formas de `images` y `labels`.

**Ayuda:** usa `DataLoader(...)` y `next(iter(...))`.

In [ ]:
BATCH_SIZE = 100

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader = DataLoader(
    valid_set,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_set,
    batch_size=BATCH_SIZE,
    shuffle=False
)

images, labels = next(iter(train_loader))

print("images.shape:", images.shape)
print("labels.shape:", labels.shape)

images.shape: torch.Size([100, 1, 28, 28])
labels.shape: torch.Size([100])


## Ejercicio 4) Comprender el **Autoencoder**

En **clasificación**:

$$
x\longrightarrow y.
$$

En un **autoencoder**:

$$
x\longrightarrow z\longrightarrow\hat{x}.
$$

El **encoder** produce una **representación latente**:

$$
z=f_{\mathrm{enc}}(x),
$$

y el **decoder** reconstruye:

$$
\hat{x}=f_{\mathrm{dec}}(z).
$$

Usaremos un *bottleneck* de dimensión 64:

$$
784\longrightarrow64\longrightarrow784.
$$

Antes de continuar, responder brevemente:
- ¿qué representa $x$?
- ¿qué representa $z$?
- ¿qué representa $\hat{x}$?
- ¿por qué 64 constituye un *bottleneck* respecto de 784?

## Ejercicio 5) Definir el autoencoder

Queremos un modelo completamente conectado con la estructura:

$$
(B,1,28,28)
\longrightarrow
(B,784)
\longrightarrow
(B,64)
\longrightarrow
(B,784)
\longrightarrow
(B,1,28,28).
$$

### Dropout opcional

El encoder incluirá `Dropout` con probabilidad configurable `p_dropout`.

**Ayudas:** completar la clase `Autoencoder`.

El **encoder** debe realizar conceptualmente:

```text
imagen → vector de 784 → capa lineal → ReLU → Dropout → código latente
```

El **decoder** debe realizar:

```text
código latente → capa lineal → Sigmoid → vector de 784
```

Finalmente, `forward()` debe recuperar la forma de imagen `(B,1,28,28)`.

**Ayudas:**
- utilizar `nn.Sequential`;
- utilizar `nn.Flatten`;
- recordar que `latent_dim` es un parámetro del constructor;
- la salida `Sigmoid` es coherente con píxeles en `[0,1]`.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=64, p_dropout=0.2):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, latent_dim),
            nn.ReLU(),
            nn.Dropout(p_dropout)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 28 * 28),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)

        # COMPLETAR:
        # Recuperar la forma de las imágenes de Fashion-MNIST.
        x_hat = x_hat.view(x.size(0), 1, 28, 28)

        return x_hat

### 5.1) Comprobar dimensiones

Si el batch contiene $B$ imágenes, verificar:

$$
(B,1,28,28)
\longrightarrow
(B,64)
\longrightarrow
(B,1,28,28).
$$

Usaremos:

```python
LATENT_DIM = 64
DROPOUT_P = 0.2
```

**Tarea:**
1. crear un modelo de ejemplo;
2. pasar el batch `images` por el modelo;
3. pasar el mismo batch solamente por el encoder;
4. imprimir las tres formas.

Se puede usar `torch.no_grad()` porque aquí sólo estamos comprobando dimensiones.

In [ ]:
LATENT_DIM = 64
DROPOUT_P = 0.2

torch.manual_seed(SEED)

# COMPLETAR
model_example = ...

with torch.no_grad():
    reconstructions = ...
    latent_codes = ...

# Descomentar para verificar:
# print("Entrada:", images.shape)
# print("Latente:", latent_codes.shape)
# print("Reconstrucción:", reconstructions.shape)

## Ejercicio 6) Función de pérdida

Usaremos el error cuadrático medio:

$$
\mathrm{MSE}(x,\hat{x})
=
\frac{1}{D}
\sum_{j=1}^{D}(x_j-\hat{x}_j)^2,
$$

con $D=784$ para una imagen de Fashion-MNIST.

**Tarea:** crea la función de pérdida de PyTorch correspondiente.

Recordar que, en un autoencoder, compararemos la reconstrucción con la propia imagen original.

In [ ]:
loss_fn = nn.MSELoss()

## Ejercicio 7) Elegir el optimizador

Trabajaremos con dos opciones:

- SGD con `lr=0.01`;
- Adam con `lr=0.001`.

En una primera ejecución se puede comenzar con Adam. Más adelante compararemos ambos usando *validation*.

**Tarea:** definir:
- `OPTIMIZER_NAME`;
- `SGD_LR`;
- `ADAM_LR`.

No crear todavía el optimizador: eso se hará en el Ejercicio 10.

In [ ]:
# COMPLETAR
OPTIMIZER_NAME = ...
SGD_LR = ...
ADAM_LR = ...

## Ejercicio 8) Implementar una época de entrenamiento

La función debe recorrer todos los *batches* de *train* y actualizar los parámetros del modelo.

El `DataLoader` devuelve `images, labels`, pero las etiquetas no son necesarias.

Orden conceptual:

```text
activar modo entrenamiento
        ↓
recorrer batches
        ↓
mover imágenes al device
        ↓
borrar gradientes anteriores
        ↓
forward
        ↓
calcular MSE
        ↓
backpropagation
        ↓
actualizar parámetros
```

**Tarea:** completar la función sin calcular métricas adicionales dentro de ella.

In [ ]:
def train_one_epoch(model, data_loader, loss_fn, optimizer, device):
    # COMPLETAR: activar modo entrenamiento.
    ...

    for images, _ in data_loader:
        # COMPLETAR: mover imágenes al dispositivo.
        images = ...

        # COMPLETAR: borrar gradientes acumulados.
        ...

        # COMPLETAR: forward.
        reconstructions = ...

        # COMPLETAR: calcular la pérdida.
        loss = ...

        # COMPLETAR: calcular gradientes.
        ...

        # COMPLETAR: actualizar parámetros.
        ...

## Ejercicio 9) Implementar la evaluación

La evaluación **no debe modificar los parámetros**.

La función debe:
1. activar modo evaluación;
2. recorrer el `DataLoader`;
3. mover las imágenes al dispositivo;
4. obtener reconstrucciones;
5. calcular MSE;
6. acumular correctamente la pérdida;
7. devolver la pérdida promedio.

Para esta guía se puede decorar la función con `@torch.no_grad()`.

Como `nn.MSELoss()` devuelve un promedio del batch, usar el tamaño del batch al acumular:

```text
pérdida media del batch × cantidad de ejemplos del batch
```

**Tarea:** completar la función.

In [ ]:
@torch.no_grad()
def evaluate(model, data_loader, loss_fn, device):
    # COMPLETAR: activar modo evaluación.
    ...

    total_loss = 0.0
    total_examples = 0

    for images, _ in data_loader:
        # COMPLETAR
        images = ...

        # COMPLETAR
        reconstructions = ...

        # COMPLETAR
        loss = ...

        # COMPLETAR
        batch_size = ...

        # COMPLETAR acumuladores.
        total_loss += ...
        total_examples += ...

    # COMPLETAR
    mean_loss = ...

    return mean_loss

## Ejercicio 10) Crear el modelo y el optimizador

Ahora utilizaremos las piezas definidas anteriormente.

**Tarea:**
1. fijar nuevamente la semilla antes de crear el modelo;
2. crear `model` con `LATENT_DIM` y `DROPOUT_P`;
3. moverlo al `device`;
4. crear SGD o Adam según `OPTIMIZER_NAME`;
5. usar `model.parameters()` para indicar qué parámetros serán optimizados.

**Ayuda:** utilizar una estructura `if / elif / else`.

In [ ]:
torch.manual_seed(SEED)

# COMPLETAR: crear el modelo y moverlo al device.
model = ...

# COMPLETAR: crear el optimizador según OPTIMIZER_NAME.
if OPTIMIZER_NAME == "SGD":
    optimizer = ...

elif OPTIMIZER_NAME == "Adam":
    optimizer = ...

else:
    raise ValueError("OPTIMIZER_NAME debe ser 'SGD' o 'Adam'.")

print("Optimizador:", OPTIMIZER_NAME)
print("Dropout p:", DROPOUT_P)

### 10.1) Reconstrucciones antes del entrenamiento

Antes de entrenar, los parámetros del modelo tienen valores iniciales aleatorios. Queremos observar qué reconstruye la red en ese estado.

Usar *validation*, no *test*.

**Tarea:** completar una función que:
1. activar modo evaluación;
2. seleccionar un *batch*;
3. conservar las primeras `n` imágenes;
4. obtener sus reconstrucciones;
5. llevar originales y reconstrucciones a CPU;
6. dibujar dos filas: originales arriba y reconstrucciones abajo.

No es necesario usar las etiquetas para calcular la reconstrucción; se pueden utilizarl sólo como títulos si así lo deseamos.

In [ ]:
@torch.no_grad()
def show_untrained_reconstructions(model, data_loader, device, n=6):
    # COMPLETAR
    ...

    # COMPLETAR: tomar un batch.
    images, labels = ...

    # COMPLETAR: conservar n ejemplos y mover imágenes al device.
    images = ...
    labels = ...

    # COMPLETAR: reconstrucciones.
    reconstructions = ...

    # COMPLETAR: llevar tensores necesarios a CPU.
    images = ...
    reconstructions = ...

    # COMPLETAR: crear figura 2 × n.
    fig, axes = ...

    for i in range(n):
        # COMPLETAR fila de originales.
        ...

        # COMPLETAR fila de reconstrucciones.
        ...

    axes[0, 0].set_ylabel("Original")
    axes[1, 0].set_ylabel("Sin entrenar")

    plt.tight_layout()
    plt.show()


# Cuando se complete la función, descomentar:
# show_untrained_reconstructions(model, valid_loader, device)

## Ejercicio 11) Entrenar el modelo

En cada época debemos:

1. ejecutar una época de entrenamiento;
2. evaluar train con los pesos finales de esa época;
3. evaluar validation con los mismos pesos;
4. guardar ambas pérdidas;
5. imprimir un resumen.

Esto permite comparar train y validation en el mismo estado del modelo.

Usar inicialmente, por ejemplo, `EPOCHS = 10`.

**Tarea:** completar el ciclo de entrenamiento.

In [ ]:
EPOCHS = 10

history = {
    "train_loss": [],
    "valid_loss": []
}

for epoch in range(1, EPOCHS + 1):
    # COMPLETAR: entrenar una época.
    ...

    # COMPLETAR: evaluar train.
    train_loss = ...

    # COMPLETAR: evaluar validation.
    valid_loss = ...

    # COMPLETAR: guardar pérdidas.
    ...

    # COMPLETAR: imprimir un resumen de la época.
    ...

## Ejercicio 12) Graficar las pérdidas

Representar las pérdidas de train y validation en función de la época.

**Tarea:** crear una figura que incluya:
- curva de train;
- curva de validation;
- nombres de ejes;
- leyenda;
- título.

**Preguntas guía:**
- ¿ambas pérdidas disminuyen?
- ¿validation sigue una tendencia semejante a train?
- ¿aparece una separación creciente entre ambas curvas?

In [ ]:
# COMPLETAR

epochs = ...

plt.figure(figsize=(7, 4))

# COMPLETAR: curvas de train y validation.
...

# COMPLETAR: etiquetas, título, leyenda y grilla.
...

plt.show()

## Ejercicio 13) Comparar SGD y Adam utilizando validation

Hasta este punto **no debemos utilizar test para elegir la configuración**.

Realizar dos experimentos.

**Experimento A:** usar

```python
OPTIMIZER_NAME = "SGD"
```

**Experimento B:** usar

```python
OPTIMIZER_NAME = "Adam"
```

Mantener iguales los demás hiperparámetros.

Valores base:

```text
SGD  → lr = 0.01
Adam → lr = 0.001
```

Para cada caso, volver a ejecutar los Ejercicios 10, 11 y 12 desde una inicialización nueva.

**Tarea:** comparar ambas configuraciones usando únicamente las pérdidas y curvas de **validation**. Anotar:
- pérdida final de validation;
- velocidad inicial de descenso;
- estabilidad de las curvas;
- optimizador elegido.

Después, dejar `OPTIMIZER_NAME` con la configuración seleccionada y volver a entrenar el modelo que será utilizado en los ejercicios siguientes.

## Ejercicio 14) Evaluar la configuración seleccionada sobre test

Recién después de seleccionar la configuración con *validation* utilizamos *test*:

$$
\text{train}
\longrightarrow
\text{aprendizaje de parámetros},
$$

$$
\text{validation}
\longrightarrow
\text{comparación de configuraciones},
$$

$$
\text{test}
\longrightarrow
\text{evaluación final}.
$$

**Tarea:** utilizar `evaluate(...)` para calcular la MSE final sobre `test_loader`.

In [ ]:
# COMPLETAR
test_loss = ...

# Descomentar al completar:
# print(f"Test MSE: {test_loss:.5f}")

## Ejercicio 15) Visualizar reconstrucciones sobre *test*

La función de pérdida (función error) MSE resume el error en un número, pero también debemos inspeccionar visualmente las reconstrucciones.

**Tarea:** completar una función que muestre:
- primera fila: imágenes originales;
- segunda fila: reconstrucciones del modelo entrenado.

Ahora sí corresponde utilizar `test_loader`, porque la configuración ya fue seleccionada.

In [ ]:
@torch.no_grad()
def show_reconstructions(model, data_loader, device, n=6):
    # COMPLETAR
    ...

    # COMPLETAR: tomar un batch y seleccionar n imágenes.
    images, _ = ...
    images = ...

    # COMPLETAR: reconstruir.
    reconstructions = ...

    # COMPLETAR: llevar a CPU.
    images = ...
    reconstructions = ...

    # COMPLETAR: figura 2 × n.
    fig, axes = ...

    for i in range(n):
        # COMPLETAR original.
        ...

        # COMPLETAR reconstrucción.
        ...

    axes[0, 0].set_ylabel("Original")
    axes[1, 0].set_ylabel("Reconstrucción")

    plt.tight_layout()
    plt.show()

# Cuando se complete la función, descomentar:
# show_reconstructions(model, test_loader, device)

## Ejercicio 16) Observar el espacio latente

El encoder transforma cada batch según:

$$
(B,1,28,28)\longrightarrow(B,64).
$$

**Tarea:**
1. tomar un batch de `test_loader`;
2. mover las imágenes al `device`;
3. activar modo evaluación;
4. sin calcular gradientes, obtener `latent_codes` utilizando sólo el encoder;
5. imprimir las formas de entrada y salida.

Si `BATCH_SIZE = 100` y `LATENT_DIM = 64`, ¿qué forma se espera para `latent_codes` antes de ejecutar la celda?

In [ ]:
# COMPLETAR: obtener un batch.
images, _ = ...

# COMPLETAR: mover imágenes.
images = ...

# COMPLETAR: activar modo evaluación.
...

with torch.no_grad():
    # COMPLETAR: obtener códigos latentes.
    latent_codes = ...

# Descomentar para verificar:
# print("Imágenes:", images.shape)
# print("Códigos latentes:", latent_codes.shape)

## Ejercicio 17) Preguntas conceptuales

Responder.

1. ¿Por qué las etiquetas de Fashion-MNIST no son necesarias para entrenar el autoencoder?
2. ¿Qué representa el código latente?
3. ¿Qué significa el bottleneck?
4. ¿Por qué usamos `MSELoss`?
5. ¿Qué hace `optimizer.zero_grad()`?
6. ¿Qué hace `loss.backward()`?
7. ¿Qué hace `optimizer.step()`?
8. ¿Por qué volvemos a evaluar train después de cada época?
9. ¿Qué diferencia existe entre `model.train()` y `model.eval()`?
10. ¿Qué hace `Dropout(p)` durante `model.train()`?
11. ¿Qué ocurre con dropout cuando ejecutamos `model.eval()`?
12. ¿Qué significa elegir `DROPOUT_P = 0.0`?
13. ¿Por qué no colocamos dropout en la capa final que reconstruye los píxeles?
14. ¿Por qué SGD y Adam usan learning rates diferentes?

## Ejercicio 18) Experimentos sugeridos

Modificar una sola variable por vez permite interpretar mejor el efecto de cada hiperparámetro.

### 18.1) Dimensión latente

Probar:

```python
LATENT_DIM = 16
LATENT_DIM = 32
LATENT_DIM = 64
LATENT_DIM = 128
```

Para cada valor registra la MSE de validation y observar las reconstrucciones.

**Pregunta guía:** ¿qué relación se observa entre el tamaño del bottleneck y la calidad de reconstrucción?

### 18.2) Dropout

Probar:

```python
DROPOUT_P = 0.0
DROPOUT_P = 0.2
DROPOUT_P = 0.5
```

Mantener constantes los demás hiperparámetros.

### 18.3) Número de épocas

Probar:

```python
EPOCHS = 5
EPOCHS = 10
EPOCHS = 20
```

Comparar las curvas de *train* y *validation*.

### 18.4) Optimizador

Comparar SGD y Adam manteniendo fija la arquitectura, el número de épocas y el *dropout*.

**Regla metodológica:** usar *validation* para comparar configuraciones. Reservar *test* para la evaluación final.